# Predictive Analytics for Student Academic Performance Using Machine Learning
## A Comparative Model Evaluation Using the xAPI-Edu-Data Dataset

**Module:** CMP600 – Dissertation  
**Dataset:** xAPI-Edu-Data (Kaggle) — 480 student records, 17 variables  
**Models:** Logistic Regression, SVM, Random Forest, XGBoost  

This notebook implements the complete machine learning pipeline:
1. Data Loading & Cleaning
2. Exploratory Data Analysis
3. Preprocessing & Encoding
4. Model Training with Hyperparameter Tuning
5. Stratified 10-Fold Cross-Validation
6. Model Comparison & Evaluation
7. Feature Importance & Explainability
8. Fairness & Sensitivity Testing
9. Model Export for Flask Dashboard Deployment


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, accuracy_score, f1_score,
                             precision_score, recall_score, confusion_matrix,
                             ConfusionMatrixDisplay)
import joblib
import json
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries imported successfully.")


## 2. Data Loading & Integrity Checks

The xAPI-Edu-Data dataset contains 480 student records with 17 variables covering demographic, behavioural, and academic factors. The target variable `Class` categorises performance into High (H), Medium (M), and Low (L).


In [ ]:
# Load dataset
df = pd.read_csv('../xAPI-Edu-Data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nTotal missing values: {df.isnull().sum().sum()}")
print(f"\nDuplicate records: {df.duplicated().sum()}")


## 3. Data Cleaning

Two duplicate records are detected and removed. No missing values require imputation. Duplicate elimination prevents artificial inflation of pattern frequency in model training (Koukaras and Tjortjis, 2025).


In [ ]:
# Remove duplicates
print(f"Records before deduplication: {len(df)}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Records after deduplication: {len(df)}")
print(f"Duplicates remaining: {df.duplicated().sum()}")


## 4. Exploratory Data Analysis

### 4.1 Target Variable Distribution


In [ ]:
# Class distribution
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = {'H': '#34D399', 'M': '#FBBF24', 'L': '#F87171'}
class_counts.plot(kind='bar', ax=axes[0], color=[colors[c] for c in class_counts.index])
axes[0].set_title('Class Distribution (Count)', fontweight='bold')
axes[0].set_xlabel('Performance Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

for i, (idx, val) in enumerate(class_counts.items()):
    axes[0].text(i, val + 3, str(val), ha='center', fontweight='bold')

axes[1].pie(class_pct, labels=[f'{c} ({v:.1f}%)' for c, v in class_pct.items()],
            colors=[colors[c] for c in class_pct.index], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Class Distribution (Percentage)', fontweight='bold')

plt.tight_layout()
plt.savefig('../models/class_distribution.png', bbox_inches='tight')
plt.show()

print(f"\nClass distribution:\n{class_counts}")
print(f"\nClass percentages:\n{class_pct.round(2)}")


### 4.2 Absence vs. Performance

In [ ]:
# Absence vs Class crosstab
ct = pd.crosstab(df['StudentAbsenceDays'], df['Class'])
ct_pct = pd.crosstab(df['StudentAbsenceDays'], df['Class'], normalize='index') * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ct.plot(kind='bar', ax=axes[0], color=[colors.get(c, '#888') for c in ct.columns])
axes[0].set_title('Absence Days vs. Performance (Count)', fontweight='bold')
axes[0].set_xlabel('Absence Category')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Class')

ct_pct.plot(kind='bar', ax=axes[1], color=[colors.get(c, '#888') for c in ct_pct.columns])
axes[1].set_title('Absence Days vs. Performance (Row %)', fontweight='bold')
axes[1].set_xlabel('Absence Category')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Class')

plt.tight_layout()
plt.savefig('../models/absence_vs_performance.png', bbox_inches='tight')
plt.show()

print("Row percentages:")
print(ct_pct.round(1))


### 4.3 Engagement Metrics by Class

In [ ]:
engagement_cols = ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion']

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

for i, col in enumerate(engagement_cols):
    order = ['H', 'M', 'L']
    sns.boxplot(data=df, x='Class', y=col, order=order, ax=axes[i],
                palette=colors, width=0.5)
    axes[i].set_title(f'{col} by Performance Class', fontweight='bold')
    axes[i].set_xlabel('Class')

plt.tight_layout()
plt.savefig('../models/engagement_boxplots.png', bbox_inches='tight')
plt.show()

print("Engagement means by class:")
print(df.groupby('Class')[engagement_cols].mean().round(1).loc[['H', 'M', 'L']])
print("\nEngagement medians by class:")
print(df.groupby('Class')[engagement_cols].median().round(1).loc[['H', 'M', 'L']])


### 4.4 Gender Distribution and Performance

In [ ]:
gender_class = pd.crosstab(df['gender'], df['Class'])
gender_class_pct = pd.crosstab(df['gender'], df['Class'], normalize='index') * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gender_class.plot(kind='bar', ax=axes[0], color=[colors.get(c, '#888') for c in gender_class.columns])
axes[0].set_title('Gender vs. Performance (Count)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

gender_class_pct.plot(kind='bar', ax=axes[1], color=[colors.get(c, '#888') for c in gender_class_pct.columns])
axes[1].set_title('Gender vs. Performance (Row %)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=0)
axes[1].set_ylabel('Percentage (%)')

plt.tight_layout()
plt.savefig('../models/gender_vs_performance.png', bbox_inches='tight')
plt.show()

print("Gender distribution:")
print(df['gender'].value_counts())
print("\nGender vs. Class (counts):")
print(gender_class)
print("\nGender vs. Class (row %):")
print(gender_class_pct.round(1))


### 4.5 Correlation Heatmap (Numerical Features)

In [ ]:
corr = df[engagement_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap of Engagement Features', fontweight='bold')
plt.tight_layout()
plt.savefig('../models/correlation_heatmap.png', bbox_inches='tight')
plt.show()


## 5. Data Preprocessing

### 5.1 Target Encoding & One-Hot Encoding

Categorical variables are converted using one-hot encoding, which preserves the categorical structure and prevents ordinal misinterpretation (Bolikulov et al., 2024).


In [ ]:
# Target encoding
le_target = LabelEncoder()
y = le_target.fit_transform(df['Class'])
class_names = le_target.classes_.tolist()
print(f"Target classes: {class_names}")
print(f"Encoding: {dict(zip(class_names, range(len(class_names))))}")

# Separate features
X = df.drop('Class', axis=1)
cat_cols = X.select_dtypes(include='object').columns.tolist()
num_cols = X.select_dtypes(include='number').columns.tolist()

print(f"\nCategorical features ({len(cat_cols)}): {cat_cols}")
print(f"Numerical features ({len(num_cols)}): {num_cols}")

# One-hot encode
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=False)
print(f"\nEncoded feature shape: {X_encoded.shape}")
print(f"Feature columns: {list(X_encoded.columns[:10])} ... ({len(X_encoded.columns)} total)")


### 5.2 Stratified Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
for i, name in enumerate(class_names):
    count = (y_train == i).sum()
    print(f"  {name}: {count} ({count/len(y_train)*100:.1f}%)")
print(f"\nTest class distribution:")
for i, name in enumerate(class_names):
    count = (y_test == i).sum()
    print(f"  {name}: {count} ({count/len(y_test)*100:.1f}%)")


## 6. Model Training with Hyperparameter Tuning

Four supervised classification models are implemented with GridSearchCV and stratified 10-fold cross-validation for hyperparameter optimisation. Grid search ensures parameters are empirically determined rather than heuristically selected (Géron, 2019).


In [ ]:
# Define models and hyperparameter grids
models_config = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'solver': ['lbfgs']
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='mlogloss', use_label_encoder=False),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.05, 0.1]
        }
    }
}

cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Train all models
results = {}
best_models = {}

for name, config in models_config.items():
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")
    
    grid = GridSearchCV(
        config['model'], config['params'],
        cv=cv_strategy, scoring='f1_weighted',
        n_jobs=-1, refit=True
    )
    grid.fit(X_train, y_train)
    
    best = grid.best_estimator_
    best_models[name] = best
    
    # Test set predictions
    y_pred = best.predict(X_test)
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average='weighted')
    f1_mac = f1_score(y_test, y_pred, average='macro')
    prec_w = precision_score(y_test, y_pred, average='weighted')
    rec_w = recall_score(y_test, y_pred, average='weighted')
    
    # Cross-validation on full dataset
    cv_scores = cross_val_score(best, X_encoded, y, cv=cv_strategy, scoring='f1_weighted')
    
    results[name] = {
        'accuracy': acc,
        'f1_weighted': f1_w,
        'f1_macro': f1_mac,
        'precision_weighted': prec_w,
        'recall_weighted': rec_w,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'cv_scores': cv_scores,
        'y_pred': y_pred,
        'best_params': grid.best_params_
    }
    
    print(f"Best parameters: {grid.best_params_}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 (weighted): {f1_w:.4f}")
    print(f"F1 (macro): {f1_mac:.4f}")
    print(f"Precision (weighted): {prec_w:.4f}")
    print(f"Recall (weighted): {rec_w:.4f}")
    print(f"CV F1 (weighted): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print("\n\nAll models trained successfully.")


## 7. Model Comparison & Evaluation

### 7.1 Summary Table


In [ ]:
# Comparison table
comparison = pd.DataFrame({
    name: {
        'Accuracy': f"{r['accuracy']:.4f}",
        'F1 (Weighted)': f"{r['f1_weighted']:.4f}",
        'F1 (Macro)': f"{r['f1_macro']:.4f}",
        'Precision (W)': f"{r['precision_weighted']:.4f}",
        'Recall (W)': f"{r['recall_weighted']:.4f}",
        'CV Mean': f"{r['cv_mean']:.4f}",
        'CV Std': f"±{r['cv_std']:.4f}",
        'Best Params': str(r['best_params'])
    }
    for name, r in results.items()
}).T

print("Model Comparison Summary:")
print(comparison.to_string())


### 7.2 Performance Bar Chart

In [ ]:
metric_names = ['Accuracy', 'F1 (Weighted)', 'Precision', 'Recall', 'CV Mean']
model_names = list(results.keys())

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(model_names))
width = 0.15

metrics_data = {
    'Accuracy': [results[m]['accuracy'] for m in model_names],
    'F1 Weighted': [results[m]['f1_weighted'] for m in model_names],
    'F1 Macro': [results[m]['f1_macro'] for m in model_names],
    'Precision': [results[m]['precision_weighted'] for m in model_names],
    'Recall': [results[m]['recall_weighted'] for m in model_names],
}

colors_bar = ['#6366F1', '#38BDF8', '#34D399', '#FBBF24', '#F87171']
for i, (metric, values) in enumerate(metrics_data.items()):
    bars = ax.bar(x + i * width, values, width, label=metric, color=colors_bar[i])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=14)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(model_names, rotation=15)
ax.legend(loc='lower right', fontsize=9)
ax.set_ylim(0.6, 0.95)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../models/model_comparison.png', bbox_inches='tight')
plt.show()


### 7.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, (name, r) in enumerate(results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=axes[i], cmap='Blues', colorbar=False)
    axes[i].set_title(f'{name}', fontweight='bold', fontsize=12)

plt.suptitle('Confusion Matrices — All Models', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../models/confusion_matrices.png', bbox_inches='tight')
plt.show()


### 7.4 Per-Class Classification Reports

In [ ]:
for name, r in results.items():
    print(f"\n{'='*50}")
    print(f"{name} — Classification Report")
    print(f"{'='*50}")
    print(classification_report(y_test, r['y_pred'], target_names=class_names))


### 7.5 Cross-Validation Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

cv_data = [results[m]['cv_scores'] for m in model_names]
bp = ax.boxplot(cv_data, labels=model_names, patch_artist=True, widths=0.5)

box_colors = ['#6366F1', '#38BDF8', '#34D399', '#FBBF24']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_ylabel('F1 Score (Weighted)')
ax.set_title('10-Fold Cross-Validation Score Distribution', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../models/cv_distribution.png', bbox_inches='tight')
plt.show()

for name in model_names:
    scores = results[name]['cv_scores']
    print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}, min={scores.min():.4f}, max={scores.max():.4f}")


## 8. Feature Importance & Explainability

Feature importance analysis identifies which variables have the strongest effect on classification outcomes. This supports the transparency requirement for educational prediction systems (Khosravi et al., 2022).


In [ ]:
# Random Forest feature importance
rf_model = best_models['Random Forest']
rf_importances = pd.Series(rf_model.feature_importances_, index=X_encoded.columns)
rf_top15 = rf_importances.nlargest(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# RF
rf_top15.sort_values().plot(kind='barh', ax=axes[0], color='#38BDF8')
axes[0].set_title('Random Forest — Top 15 Features', fontweight='bold')
axes[0].set_xlabel('Gini Importance')

# XGBoost
xgb_model = best_models['XGBoost']
xgb_importances = pd.Series(xgb_model.feature_importances_, index=X_encoded.columns)
xgb_top15 = xgb_importances.nlargest(15)

xgb_top15.sort_values().plot(kind='barh', ax=axes[1], color='#34D399')
axes[1].set_title('XGBoost — Top 15 Features', fontweight='bold')
axes[1].set_xlabel('Feature Importance')

plt.tight_layout()
plt.savefig('../models/feature_importance.png', bbox_inches='tight')
plt.show()

print("Random Forest Top 10 Features:")
for feat, imp in rf_importances.nlargest(10).items():
    print(f"  {feat}: {imp:.4f}")

print("\nXGBoost Top 10 Features:")
for feat, imp in xgb_importances.nlargest(10).items():
    print(f"  {feat}: {imp:.4f}")


## 9. Fairness & Sensitivity Testing

### 9.1 Gender Subgroup Analysis

Subgroup evaluation determines whether predictive performance is equitable across demographic groups. Fairness-conscious machine learning requires subgroup comparison as a quantifiable model-suitability criterion (Kesgin et al., 2025).


In [ ]:
# Use Random Forest (best model) for fairness analysis
rf_pred = results['Random Forest']['y_pred']
test_df = df.iloc[X_test.index].copy()
test_df['y_true'] = y_test
test_df['y_pred'] = rf_pred

print("Gender Subgroup Analysis (Random Forest)")
print("=" * 55)

fairness_results = {}
for gender in ['M', 'F']:
    mask = test_df['gender'] == gender
    sub = test_df[mask]
    
    f1 = f1_score(sub['y_true'], sub['y_pred'], average='weighted')
    acc = accuracy_score(sub['y_true'], sub['y_pred'])
    rec = recall_score(sub['y_true'], sub['y_pred'], average='weighted')
    prec = precision_score(sub['y_true'], sub['y_pred'], average='weighted')
    
    gender_label = 'Male' if gender == 'M' else 'Female'
    fairness_results[gender] = {'f1': f1, 'accuracy': acc, 'recall': rec, 'precision': prec, 'count': len(sub)}
    
    print(f"\n{gender_label} (n={len(sub)}):")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  F1 (W):    {f1:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")

f1_gap = abs(fairness_results['M']['f1'] - fairness_results['F']['f1'])
print(f"\nF1 Gap (Male - Female): {f1_gap:.4f}")
print(f"Assessment: {'Minimal disparity — acceptable fairness' if f1_gap < 0.05 else 'Significant disparity — requires mitigation'}")


### 9.2 Sensitivity Analysis — Removing Gender Features

In [ ]:
# Retrain RF without gender features
gender_cols = [c for c in X_encoded.columns if c.startswith('gender_')]
X_no_gender = X_encoded.drop(columns=gender_cols)

X_train_ng = X_no_gender.iloc[X_train.index]
X_test_ng = X_no_gender.iloc[X_test.index]

rf_ng = RandomForestClassifier(**best_models['Random Forest'].get_params())
rf_ng.fit(X_train_ng, y_train)
y_pred_ng = rf_ng.predict(X_test_ng)

f1_with = f1_score(y_test, rf_pred, average='weighted')
f1_without = f1_score(y_test, y_pred_ng, average='weighted')
f1_drop = f1_with - f1_without

print("Sensitivity Analysis: Gender Feature Removal")
print("=" * 50)
print(f"F1 (with gender):    {f1_with:.4f}")
print(f"F1 (without gender): {f1_without:.4f}")
print(f"F1 drop:             {f1_drop:.4f} ({f1_drop*100:.1f}%)")
print(f"\nConclusion: {'Model relies primarily on behavioural features — minimal demographic dependency' if abs(f1_drop) < 0.05 else 'Significant dependency on gender features detected'}")

# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['With Gender', 'Without Gender'], [f1_with, f1_without],
              color=['#38BDF8', '#FBBF24'], width=0.4)
ax.set_ylabel('F1 Score (Weighted)')
ax.set_title('Sensitivity Analysis: Impact of Gender Feature Removal', fontweight='bold')
ax.set_ylim(0.7, 0.9)
for bar, val in zip(bars, [f1_with, f1_without]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../models/sensitivity_analysis.png', bbox_inches='tight')
plt.show()


## 10. Export Best Model for Flask Dashboard

The best-performing model (Random Forest) is saved using `joblib` along with necessary metadata. The Flask web application will load these artefacts to serve real-time predictions through the early-warning dashboard.


In [ ]:
# Save best model (Random Forest)
output_dir = '../models'
os.makedirs(output_dir, exist_ok=True)

# Save model
joblib.dump(best_models['Random Forest'], os.path.join(output_dir, 'best_model.pkl'))
print("Saved: best_model.pkl (Random Forest)")

# Save label encoder
joblib.dump(le_target, os.path.join(output_dir, 'label_encoder.pkl'))
print("Saved: label_encoder.pkl")

# Save feature columns (order matters for prediction)
feature_cols = list(X_encoded.columns)
joblib.dump(feature_cols, os.path.join(output_dir, 'feature_columns.pkl'))
print(f"Saved: feature_columns.pkl ({len(feature_cols)} features)")

# Save metadata for the Flask app
metadata = {
    'class_names': class_names,
    'categorical_columns': cat_cols,
    'numerical_columns': num_cols,
    'best_model_name': 'Random Forest',
    'best_model_params': {k: str(v) for k, v in best_models['Random Forest'].get_params().items()},
    'test_accuracy': float(results['Random Forest']['accuracy']),
    'test_f1_weighted': float(results['Random Forest']['f1_weighted']),
    'cv_mean': float(results['Random Forest']['cv_mean']),
    'cv_std': float(results['Random Forest']['cv_std']),
    'feature_importance': {str(k): float(v) for k, v in rf_importances.nlargest(15).items()},
    'model_results_summary': {
        name: {
            'accuracy': float(r['accuracy']),
            'f1_weighted': float(r['f1_weighted']),
            'cv_mean': float(r['cv_mean'])
        } for name, r in results.items()
    },
    'fairness': {
        g: {k: float(v) if isinstance(v, (float, np.floating)) else v
            for k, v in data.items()}
        for g, data in fairness_results.items()
    },
    'sensitivity': {
        'with_gender_f1': float(f1_with),
        'without_gender_f1': float(f1_without),
        'f1_drop': float(f1_drop)
    }
}

with open(os.path.join(output_dir, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print("Saved: metadata.json")

# Save the original dataframe columns for the Flask form
original_cols_info = {}
for col in cat_cols:
    original_cols_info[col] = sorted(df[col].unique().tolist())

with open(os.path.join(output_dir, 'column_options.json'), 'w') as f:
    json.dump(original_cols_info, f, indent=2)
print("Saved: column_options.json")

print("\n" + "="*50)
print("ALL ARTEFACTS EXPORTED SUCCESSFULLY")
print("="*50)
print(f"\nFiles in {output_dir}/:")
for f_name in sorted(os.listdir(output_dir)):
    size = os.path.getsize(os.path.join(output_dir, f_name))
    print(f"  {f_name} ({size:,} bytes)")
print("\nThe Flask app (app.py) will load these files to serve predictions.")
